In [1]:
import argparse
import os
import cv2 
import glob
# import skvideo.io

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 

from easydict import EasyDict

import torch
from torch.utils.data import DataLoader

from posetail.datasets.datasets import Rat7mDataset, Rat7mIterableDataset, custom_collate_2d
from train_utils import *

%load_ext autoreload
%autoreload 2

In [4]:
# testing iterable dataset
# run_id = 'run-20250303_121158-17z0mb1w'
run_id = 'run-20250423_201321-kmadlx6h'
config_path = f'/allen/aind/scratch/katie.rupp/wandb/{run_id}/files/config.toml'
config = load_config(config_path)
device = (torch.device('cuda') if torch.cuda.is_available() else 'cpu')

video_path = '/allen/aind/scratch/katie.rupp/data/rat7m/videos/s5-d2/s5-d2-camera1-0.mp4'
data_path = '/allen/aind/scratch/katie.rupp/data/rat7m/data/mocap-s5-d2.mat'

# iterable dataset
dataset = get_dataset(**config.dataset.train)

# non-iterable dataset
dataset = Rat7mDataset(
    video_path = video_path, 
    data_path = data_path, 
    n_frames = config.dataset.test.n_frames)

dataloader = DataLoader(
    dataset, 
    batch_size = config.dataset.batch_size, 
    collate_fn = custom_collate_2d)

for j, batch in enumerate(dataloader):
    views = [view.to(device) for view in batch.views]
    coords = batch.coords.to(device)
    fnum = batch.fnums.to(device)
    break

In [ ]:
video_path = '/allen/aind/scratch/katie.rupp/data/rat7m/videos/s5-d2/s5-d2-camera1-0.mp4'
results_path = '/home/katie.rupp/posetail/results/s5-d2-camera1-0_predictions.npz'
device = (torch.device('cuda') if torch.cuda.is_available() else 'cpu')
coords_pred, vis_pred, conf_pred, coords_true, vis_true, fnums, video_path = load_predictions(results_path, device)
fig_path = safe_make('figures')

coords_true = coords_true.cpu().numpy().astype(int)
coords_pred = coords_pred.cpu().numpy().astype(int)

i = 0
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = round(cap.get(cv2.CAP_PROP_FPS), 2)

outname = os.path.join(fig_path, 'output2.avi')
ret = True

writer = skvideo.io.FFmpegWriter(outname, inputdict={
        # '-hwaccel': 'auto',
        '-framerate': str(fps),
    }, outputdict={
        '-vcodec': 'h264', '-qp': '28',
        '-pix_fmt': 'yuv420p', # to support more players
        '-vf': 'pad=ceil(iw/2)*2:ceil(ih/2)*2' # to handle width/height not divisible by 2
})

while ret:

    ret, frame = cap.read() 
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    if i not in fnums:
        writer.writeFrame(frame)
        continue
    
    if not ret: 
        break

    for coord in coords_true[i]:
        cv2.circle(frame, tuple(coord), 5, (0, 255, 0), -1)

    for coord in coords_pred[i]: 
        cv2.circle(frame, tuple(coord), 5, (255, 0, 0), -1)

    # outpath = os.path.join(fig_path, f'out{i}.png')
    # cv2.imwrite(os.path.join(fig_path, f'out{i}.png'), frame)
    writer.writeFrame(frame)
    i += 1

cap.release()
writer.close()
